# Train Zero-DCE on Colab

This notebook only sets up the environment and calls `src/train.py`. All training logic lives in the repository.

**Before running:**
1. Push the repository to GitHub and set `REPO_URL` below.
2. Upload `LOL.zip` (a zip whose top-level folder is `LOL/` containing `our485/` and `eval15/`) to Google Drive at `MyDrive/low-light-enhancement/LOL.zip`.
3. *Runtime → Change runtime type → T4 GPU*.

Checkpoints are written to Google Drive. If the session disconnects, run all cells again: training resumes from `last.pth`.

In [ ]:
REPO_URL = "https://github.com/<your-username>/low-light-enhancement.git"
DRIVE_DIR = "/content/drive/MyDrive/low-light-enhancement"
REPO_DIR = "/content/low-light-enhancement"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

Colab already ships a CUDA build of PyTorch. `requirements.txt` pins the CPU-oriented versions used locally, so torch and torchvision are skipped here to keep Colab's GPU build.

In [ ]:
!grep -v -E '^(torch|torchvision)==' requirements.txt > /tmp/requirements-colab.txt
!pip install -q -r /tmp/requirements-colab.txt
!python -c "import torch; print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())"

In [ ]:
!mkdir -p data && unzip -q -o {DRIVE_DIR}/LOL.zip -d data/
!find data/LOL -name '*.png' | cut -d/ -f3,4 | sort | uniq -c

In [ ]:
!python -m src.train --config config.yaml --output-dir {DRIVE_DIR}/checkpoints

In [ ]:
from IPython.display import Image

Image(f"{DRIVE_DIR}/checkpoints/loss_curves.png")

**After training**, download these files from `MyDrive/low-light-enhancement/checkpoints/`:

- `zerodce.pth`: best weights by validation PSNR (about 320 KB). Place it at `weights/zerodce.pth` in the repository.
- `history.csv` and `loss_curves.png`: the training log and curves.

`last.pth` and `best.pth` are full checkpoints for resuming and do not need to be committed.